In [31]:
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Set, Tuple
from collections import defaultdict
from dataclasses import dataclass, asdict
import pandas as pd
import datetime
import pickle


In [57]:
GROUND_TRUTH_PATH = Path("../data/ir_ground_truth.json")
RETRIEVAL_DIR = Path("../retrieval/VD/e5-intertext")
OUTPUT_DIR = Path("../evaluation/")

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Model name
MODEL_NAME = "e5-intertext"

# K values
K_VALUES = [5,10,20]

In [58]:
def load_ground_truth(filepath: Path) -> Tuple[Dict[str, Set[str]], Dict, Dict[str, str], Dict[str, str]]:
    print(f"Loading ground truth from: {filepath.name}")
    
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    metadata = data.get("metadata", {})
    annotations = data.get("ground_truth", [])
    
    # Build lookup dictionaries
    ground_truth_dict = {}
    query_to_type = {}
    query_to_father = {}
    
    for annotation in annotations:
        query_id = annotation["query_chunk_id"]
        relevant_chunks = set(annotation["relevant_chunks"])
        
        ground_truth_dict[query_id] = relevant_chunks
        query_to_type[query_id] = annotation.get("reference_type", "unknown")
        query_to_father[query_id] = annotation.get("church_fathers", "unknown")
    
    print(f" Loaded {len(ground_truth_dict)} annotated queries")
    print(f" Explicit: {metadata.get('by_reference_type', {}).get('explicit', 0)}")
    print(f" Implicit: {metadata.get('by_reference_type', {}).get('implicit', 0)}")
    print()
    
    return ground_truth_dict, metadata, query_to_type, query_to_father


# Load ground truth
ground_truth, gt_metadata, query_to_type, query_to_father = load_ground_truth(GROUND_TRUTH_PATH)

print(f"Total annotated queries: {len(ground_truth)}")

Loading ground truth from: ir_ground_truth.json
 Loaded 100 annotated queries
 Explicit: 50
 Implicit: 50

Total annotated queries: 100


In [60]:
# Load from pickle
retrieval_path = RETRIEVAL_DIR / f"retrieval_results_{MODEL_NAME.lower()}.pkl"

print(f"Loading from: {retrieval_path}")

with open(retrieval_path, 'rb') as f:
    retrieval_results = pickle.load(f)

print(f"  Loaded retrieval results")
print(f"  Type: {type(retrieval_results)}")
print(f"  Number of queries: {len(retrieval_results)}")

# Show structure
sample_query = list(retrieval_results.keys())[0]
sample_candidates = retrieval_results[sample_query][:3]

print(f"\nSample query: {sample_query}")
print(f"  Top 3 candidates:")
for rank, (cand_id, score) in enumerate(sample_candidates, 1):
    print(f"    {rank}. {cand_id}: {score:.4f}")

Loading from: ../retrieval/VD/e5-intertext/retrieval_results_e5-intertext.pkl
  Loaded retrieval results
  Type: <class 'dict'>
  Number of queries: 19466

Sample query: 10015_sent_0
  Top 3 candidates:
    1. 033_Augustinus-Hipponensis_Epistolae_window_3477: 0.7647
    2. 022_Hieronymus-Stridonensis_Epistolae_window_1738: 0.7571
    3. 066_Benedictus-Nursiae_Regula-cum-commentariis_window_3169: 0.7568


In [61]:
def parse_chunk_id(chunk_id: str):
    """
    Parse a chunk_id into (source_file, sentence_indices).
    Handles both:
      - BDC format:  '10286_sent_40'       -> ('10286', {40})
      - BDC format:  '10286_sent_40_42'    -> ('10286', {40, 41, 42})
      - PSC format:  '022_Hieronymus-Stridonensis_Epistolae_window_1994'
                     -> treat as opaque (no sentence_indices)
    Returns (source_prefix: str, indices: set[int] | None)
    """
    parts = chunk_id.split('_')
    # BDC chunks always have 'sent' as the second-to-last or third token
    try:
        sent_idx = parts.index('sent')
        source_prefix = '_'.join(parts[:sent_idx])
        position_parts = parts[sent_idx + 1:]
        if len(position_parts) == 1:
            indices = {int(position_parts[0])}
        elif len(position_parts) == 2:
            start, end = int(position_parts[0]), int(position_parts[1])
            indices = set(range(start, end + 1))
        else:
            indices = None
        return source_prefix, indices
    except (ValueError, IndexError):
        # PSC = no sentence_indices available
        return chunk_id, None


def chunks_match(retrieved_id: str, gt_id: str) -> bool:
    """
    Containment-aware match between a retrieved chunk and a ground-truth chunk.
    A hit is counted if:
      1. Exact ID match, OR
      2. Same source file AND sentence indices overlap (either contains the other)
    """
    if retrieved_id == gt_id:
        return True
    src_r, idx_r = parse_chunk_id(retrieved_id)
    src_g, idx_g = parse_chunk_id(gt_id)
    if src_r != src_g:
        return False
    if idx_r is None or idx_g is None:
        return False
    # Hit if one set of indices is a subset of the other
    return idx_r.issubset(idx_g) or idx_g.issubset(idx_r)


def count_hits(retrieved_ids: list, relevant_set: set) -> int:
    """
    Count how many items in retrieved_ids match any chunk in relevant_set,
    using containment-aware matching. Each relevant chunk can only be
    matched once (to avoid double-counting sentence + window twins).
    """
    matched_gt = set()  # track which GT chunks have already been matched
    hits = 0
    for ret_id in retrieved_ids:
        for gt_id in relevant_set:
            if gt_id not in matched_gt and chunks_match(ret_id, gt_id):
                hits += 1
                matched_gt.add(gt_id)
                break
    return hits


def evaluate_retrieval(
    retrieval_results: Dict[str, List[Tuple[str, float]]],
    ground_truth: Dict[str, Set[str]],
    query_to_type: Dict[str, str],
    query_to_father: Dict[str, str],
    k_values: List[int] = K_VALUES
) -> Dict:
    """
    Evaluate retrieval results with breakdown by reference type and Church Father.
    Uses containment-aware matching: a retrieved chunk counts as a hit if its
    sentence indices overlap (in either direction) with the ground-truth chunk
    from the same source file.

    Args:
        retrieval_results: {query_id: [(candidate_id, score), ...]}
        ground_truth: {query_id: {relevant_candidate_ids}}
        query_to_type: {query_id: 'explicit'/'implicit'}
        query_to_father: {query_id: 'P18700' (Patrologia ID)}
        k_values: List of k values to evaluate

    Returns:
        Dict with overall, by_reference_type, and by_church_father results
    """

    # Church Father ID to name mapping
    FATHER_NAMES = {
        'P18700': 'Augustinus',
        'P17746': 'Tertullianus',
        'P18993': 'Ambrosius',
        'P18986': 'Hieronymus',
        'P18048': 'Hilarius',
        'P18988': 'Cyprian'
    }

    results = {}

    for k in k_values:
        print(f"\nEvaluating k={k}...")

        # Overall metrics
        precisions = []
        recalls = []
        f1_scores = []
        reciprocal_ranks = []

        # Metrics by reference type
        explicit_recalls = []
        explicit_mrr = []
        implicit_recalls = []
        implicit_mrr = []

        # Metrics by Church Father
        father_recalls = {}  # {father_id: [recall_values]}
        father_mrr = {}
        father_counts = {}

        queries_evaluated = 0
        queries_skipped = 0

        explicit_count = 0
        implicit_count = 0

        for query_id, relevant_set in ground_truth.items():
            ref_type = query_to_type.get(query_id, 'unknown')
            father_id = query_to_father.get(query_id, 'unknown')

            # Initialize father tracking
            if father_id not in father_recalls:
                father_recalls[father_id] = []
                father_mrr[father_id] = []
                father_counts[father_id] = 0

            # Skip if query not in retrieval results
            if query_id not in retrieval_results:
                queries_skipped += 1
                continue

            queries_evaluated += 1
            father_counts[father_id] += 1

            # Count by type
            if ref_type == 'explicit':
                explicit_count += 1
            elif ref_type == 'implicit':
                implicit_count += 1

            # Get top-k candidates
            retrieved = retrieval_results[query_id][:k]
            retrieved_ids = [cand_id for cand_id, score in retrieved]

            # Containment-aware hit counting
            true_positives = count_hits(retrieved_ids, relevant_set)

            precision = true_positives / k if k > 0 else 0
            recall = true_positives / len(relevant_set) if len(relevant_set) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            precisions.append(precision)
            recalls.append(recall)
            f1_scores.append(f1)

            # MRR: rank of first containment-aware hit
            rr = 0.0
            for rank, (cand_id, score) in enumerate(retrieved, 1):
                if any(chunks_match(cand_id, gt_id) for gt_id in relevant_set):
                    rr = 1.0 / rank
                    break

            reciprocal_ranks.append(rr)

            # Store by reference type
            if ref_type == 'explicit':
                explicit_recalls.append(recall)
                explicit_mrr.append(rr)
            elif ref_type == 'implicit':
                implicit_recalls.append(recall)
                implicit_mrr.append(rr)

            # Store by Church Father
            father_recalls[father_id].append(recall)
            father_mrr[father_id].append(rr)

        # Compute averages by Church Father
        father_results = {}
        for father_id in father_recalls.keys():
            if father_recalls[father_id]:  # Has data
                father_results[father_id] = {
                    'name': FATHER_NAMES.get(father_id, father_id),
                    'recall': np.mean(father_recalls[father_id]),
                    'mrr': np.mean(father_mrr[father_id]),
                    'count': father_counts[father_id]
                }

        # Average metrics
        results[k] = {
            'overall': {
                'precision': np.mean(precisions) if precisions else 0.0,
                'recall': np.mean(recalls) if recalls else 0.0,
                'f1': np.mean(f1_scores) if f1_scores else 0.0,
                'mrr': np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0,
                'num_queries_evaluated': queries_evaluated,
                'num_queries_skipped': queries_skipped
            },
            'by_reference_type': {
                'explicit': {
                    'recall': np.mean(explicit_recalls) if explicit_recalls else 0.0,
                    'mrr': np.mean(explicit_mrr) if explicit_mrr else 0.0,
                    'count': explicit_count
                },
                'implicit': {
                    'recall': np.mean(implicit_recalls) if implicit_recalls else 0.0,
                    'mrr': np.mean(implicit_mrr) if implicit_mrr else 0.0,
                    'count': implicit_count
                }
            },
            'by_church_father': father_results
        }

        # Print overall results
        print(f"  Overall:")
        print(f"    Recall@{k}: {results[k]['overall']['recall']:.4f}")
        print(f"    MRR: {results[k]['overall']['mrr']:.4f}")
        print(f"    Precision: {results[k]['overall']['precision']:.4f}")
        print(f"    F1: {results[k]['overall']['f1']:.4f}")
        print(f"    Queries evaluated: {queries_evaluated}/{len(ground_truth)}")

        # Print breakdown by reference type
        print(f"\n  By Reference Type:")
        print(f"    Explicit (n={explicit_count}):")
        print(f"      Recall@{k}: {results[k]['by_reference_type']['explicit']['recall']:.4f}")
        print(f"      MRR: {results[k]['by_reference_type']['explicit']['mrr']:.4f}")
        print(f"    Implicit (n={implicit_count}):")
        print(f"      Recall@{k}: {results[k]['by_reference_type']['implicit']['recall']:.4f}")
        print(f"      MRR: {results[k]['by_reference_type']['implicit']['mrr']:.4f}")

        # Print breakdown by Church Father (top 5 by count)
        print(f"\n  By Church Father (Top 5):")
        sorted_fathers = sorted(
            father_results.items(),
            key=lambda x: x[1]['count'],
            reverse=True
        )
        for father_id, metrics in sorted_fathers[:5]:
            print(f"    {metrics['name']} (n={metrics['count']}):")
            print(f"      Recall@{k}: {metrics['recall']:.4f}")
            print(f"      MRR: {metrics['mrr']:.4f}")

        if queries_skipped > 0:
            print(f"Queries skipped: {queries_skipped}")

    return results


In [62]:
print(f"Model: {MODEL_NAME}")
print(f"Ground truth entries: {len(ground_truth)}")
print(f"K values: {K_VALUES}")
print()

eval_results = evaluate_retrieval(
    retrieval_results=retrieval_results,
    ground_truth=ground_truth,
    query_to_type=query_to_type,
    query_to_father=query_to_father, 
    k_values=K_VALUES
)

Model: e5-intertext
Ground truth entries: 100
K values: [5, 10, 20]


Evaluating k=5...
  Overall:
    Recall@5: 0.3300
    MRR: 0.2615
    Precision: 0.0660
    F1: 0.1100
    Queries evaluated: 100/100

  By Reference Type:
    Explicit (n=50):
      Recall@5: 0.6200
      MRR: 0.5090
    Implicit (n=50):
      Recall@5: 0.0400
      MRR: 0.0140

  By Church Father (Top 5):
    Augustinus (n=54):
      Recall@5: 0.2963
      MRR: 0.2065
    Hieronymus (n=12):
      Recall@5: 0.3333
      MRR: 0.2917
    Tertullianus (n=9):
      Recall@5: 0.4444
      MRR: 0.3889
    Hilarius (n=5):
      Recall@5: 0.8000
      MRR: 0.7000
    Cyprian (n=4):
      Recall@5: 0.2500
      MRR: 0.2500

Evaluating k=10...
  Overall:
    Recall@10: 0.3800
    MRR: 0.2679
    Precision: 0.0380
    F1: 0.0691
    Queries evaluated: 100/100

  By Reference Type:
    Explicit (n=50):
      Recall@10: 0.7000
      MRR: 0.5192
    Implicit (n=50):
      Recall@10: 0.0600
      MRR: 0.0165

  By Church Father (T

In [63]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# output data with breakdown
output_data = {
    "metadata": {
        "model_name": MODEL_NAME,
        "evaluation_date": str(pd.Timestamp.now()),
        "ground_truth_size": len(ground_truth),
        "k_values": K_VALUES
    },
    "results": {}
}

for k, metrics in eval_results.items():
    output_data["results"][f"k{k}"] = {
        "overall": {
            "precision": float(metrics['overall']['precision']),
            "recall": float(metrics['overall']['recall']),
            "f1": float(metrics['overall']['f1']),
            "mrr": float(metrics['overall']['mrr']),
            "num_queries_evaluated": int(metrics['overall']['num_queries_evaluated'])
        },
        "by_reference_type": {
            "explicit": {
                "recall": float(metrics['by_reference_type']['explicit']['recall']),
                "mrr": float(metrics['by_reference_type']['explicit']['mrr']),
                "count": int(metrics['by_reference_type']['explicit']['count'])
            },
            "implicit": {
                "recall": float(metrics['by_reference_type']['implicit']['recall']),
                "mrr": float(metrics['by_reference_type']['implicit']['mrr']),
                "count": int(metrics['by_reference_type']['implicit']['count'])
            }
        }
    }


# Create summary DataFrame
summary_data = []
for k in eval_results.keys():
    overall = eval_results[k]['overall']
    explicit = eval_results[k]['by_reference_type']['explicit']
    implicit = eval_results[k]['by_reference_type']['implicit']
    
    summary_data.append({
        'k': k,
        'Overall_Recall': overall['recall'],
        'Overall_MRR': overall['mrr'],
        'Explicit_Recall': explicit['recall'],
        'Explicit_MRR': explicit['mrr'],
        'Explicit_N': explicit['count'],
        'Implicit_Recall': implicit['recall'],
        'Implicit_MRR': implicit['mrr'],
        'Implicit_N': implicit['count']
    })

results_df = pd.DataFrame(summary_data)

# Save CSV
csv_path = output_dir / f"evaluation_results_{MODEL_NAME.lower()}.csv"
results_df.to_csv(csv_path, index=False)

print(f"\nResults summary:")
print(results_df.to_string(index=False))


Results summary:
 k  Overall_Recall  Overall_MRR  Explicit_Recall  Explicit_MRR  Explicit_N  Implicit_Recall  Implicit_MRR  Implicit_N
 5            0.33     0.261500             0.62      0.509000          50             0.04      0.014000          50
10            0.38     0.267857             0.70      0.519214          50             0.06      0.016500          50
20            0.40     0.268910             0.72      0.520267          50             0.08      0.017553          50


In [13]:
# Save Church Father Recall@10 to one CSV (one column per model)
FATHER_RECALL_CSV = OUTPUT_DIR / "church_father_recall_at10.csv"

# Extract Recall@10 per church father from eval_results
father_data = eval_results[10]['by_church_father']
new_col = {metrics['name']: round(metrics['recall'], 4) for metrics in father_data.values()}

# Load existing CSV or create a fresh DataFrame
if FATHER_RECALL_CSV.exists():
    cf_df = pd.read_csv(FATHER_RECALL_CSV, index_col='church_father')
else:
    cf_df = pd.DataFrame()
    cf_df.index.name = 'church_father'

# Add / overwrite column for the current model
cf_df[MODEL_NAME] = pd.Series(new_col)

# Ensure index name is set after assignment
cf_df.index.name = 'church_father'
cf_df = cf_df.sort_index()

# Save back
cf_df.to_csv(FATHER_RECALL_CSV)

print(f"Church Father Recall@10 saved to: {FATHER_RECALL_CSV}")
print(cf_df.to_string())

Church Father Recall@10 saved to: ../evaluation/church_father_recall_at10.csv
                   e5  laberta_vulg_philberta_reranked  laberta_vulg-synt  sphilberta  sphilberta-intertext  locisimiles_k100  laberta_vulg  e5-intertext  locisimiles_e5-phil-v2
church_father                                                                                                                                                                    
Ambrosius      0.3333                           0.3333             0.0000      0.0000                0.0000            0.0000        0.0000        0.3333                  0.0000
Augustinus     0.2407                           0.3148             0.2593      0.1111                0.1852            0.3333        0.2593        0.3333                  0.3333
Cyprian        0.2500                           0.5000             0.5000      0.0000                0.0000            0.0000        0.5000        0.2500                  0.5000
Hieronymus     0.4167           